In [1]:
import os
import requests
import torch
from PIL import Image
import soundfile
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig

model_id = 'microsoft/Phi-4-multimodal-instruct'
model_path = '/home/mvnl/code/phi4/output'
kwargs = {}
kwargs['torch_dtype'] = torch.bfloat16

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
print(processor.tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype='auto',
    _attn_implementation='flash_attention_2',
).cuda()
print("model.config._attn_implementation:", model.config._attn_implementation)

generation_config = GenerationConfig.from_pretrained(model_path, 'generation_config.json')

user_prompt = '<|user|>'
assistant_prompt = '<|assistant|>'
prompt_suffix = '<|end|>'

/home/mvnl/miniconda3/envs/llama_factory/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mvnl/miniconda3/envs/llama_factory/lib/python3.11/site-packages/transformers/models/auto/image_processing_auto.py:590: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


GPT2TokenizerFast(name_or_path='microsoft/Phi-4-multimodal-instruct', vocab_size=200019, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	199999: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200010: AddedToken("<|endoftext10|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200011: AddedToken("<|endoftext11|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200018: AddedToken("<|endofprompt|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	200019: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	200020: AddedToken("<|end|>", rstrip=T

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
/home/mvnl/.cache/huggingface/modules/transformers_modules/output/speech_conformer_encoder.py:2775: FutureWarning: Please specify CheckpointImpl.NO_REENTRANT as CheckpointImpl.REENTRANT will soon be removed as the default and eventually deprecated.
  lambda i: encoder_checkpoint_wrapper(
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  7.10it/s]
Some weights of Phi4MMForCausalLM were not initialized from the model checkpoint at /home/mvnl/code/phi4/output and are newly initialized: ['model.embed_tokens_extend.audio_embed.audio_projection.speech.0.bias', 'model.embed_tokens_extend.audio_embed.audio_projection.speech.0.weight', 'model.embed_tokens_extend.audio_embed.audio_projection.speech.2.bias', 'model.embed_tokens_extend.audio_embed.audio_projection.speech.2.weight', 'model.embed_tokens_extend.audio_e

model.config._attn_implementation: flash_attention_2


In [2]:
def inference(AOI_path,AOI_number):
    chat = [
        {
        'role': 'user',
        'content': f"<|image_1|>This is a course slide imgae that contains  Areas of Interest (AOI) and I will give you later. Please follow the instructions below when responding:\n\n. When you receive an image containing an AOI, carefully observe and describe the content and characteristics of that area.\n Your description should be detailed and clear, highlighting the key elements in the image.",
        },
        {
            'role': 'assistant',
            'content': "Okay, I received the lecture slides. Please provide image of area of interest (AOI) and I will describe it as you request.",
        },
        {'role': 'user', 'content': f'<|image_2|>Please describe this AOI.'},
    ]
    original = Image.open(f'{AOI_path}/original.jpg')
    image = Image.open(f'{AOI_path}/img/{AOI_number}.jpg')
    prompt = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    # need to remove last <|endoftext|> if it is there, which is used for training, not inference. For training, make sure to add <|endoftext|> in the end.
    if prompt.endswith('<|endoftext|>'):
        prompt = prompt.rstrip('<|endoftext|>')
    #print(prompt)
    inputs = processor(prompt, [original,image], return_tensors='pt').to('cuda:0')
    generate_ids = model.generate(
        **inputs,
        max_new_tokens=1000,
        generation_config=generation_config,
    )
    generate_ids = generate_ids[:, inputs['input_ids'].shape[1] :]
    response = processor.batch_decode(
        generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return response

In [14]:
def inference_batch(AOI_paths,AOI_numbers,max_heigh):
    chat = [
        {
        'role': 'user',
        'content': f"<|image_1|>This is a course slide imgae that contains  Areas of Interest (AOI) and I will give you later. Please follow the instructions below when responding:\n\n. When you receive an image containing an AOI, carefully observe and describe the content and characteristics of that area.\n Your description should be detailed and clear, highlighting the key elements in the image.",
        },
        {
            'role': 'assistant',
            'content': "Okay, I received the lecture slides. Please provide image of area of interest (AOI) and I will describe it as you request.",
        },
        {'role': 'user', 'content': f'<|image_2|>Please describe this AOI.'},
    ]
    input_images = []
    for i in range(len(AOI_paths)):
        AOI_path = AOI_paths[i]
        AOI_number = AOI_numbers[i]
        original = Image.open(f'{AOI_path}/original.jpg')
        image = Image.open(f'{AOI_path}/img/{AOI_number}.jpg')
        #if original.size[1] > max_heigh:
        #    scale = max_heigh / original.size[1] 
        #    original = original.resize((int(original.size[0] * scale),int(original.size[1] * scale)))
        #    image = original.resize((int(image.size[0] * scale),int(image.size[1] * scale)))
        input_images += [original,image]
    prompt = processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    # need to remove last <|endoftext|> if it is there, which is used for training, not inference. For training, make sure to add <|endoftext|> in the end.
    if prompt.endswith('<|endoftext|>'):
        prompt = prompt.rstrip('<|endoftext|>')
    #print(prompt)
    inputs = processor([prompt] * len(AOI_numbers), input_images, return_tensors='pt').to('cuda:0')
    generate_ids = model.generate(
        **inputs,
        max_new_tokens=1000,
        generation_config=generation_config,
    )
    generate_ids = generate_ids[:, inputs['input_ids'].shape[1] :]
    response = processor.batch_decode(
        generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return response

In [ ]:
base_path = '/home/mvnl/code/DSA/course/B1_string_matching/AOIs'
for video in os.listdir(base_path):
    for page in os.listdir(f'{base_path}/{video}'):
        for AOI in os.listdir(f'{base_path}/{video}/{page}/img'):
            response = inference(f'{base_path}/{video}/{page}',AOI.replace('.jpg',''))
            

NameError: name 'os' is not defined